# TabPFN Quickstart

*A first look at making predictions on tabular data with TabPFN.*

TabPFN is a foundation model for tabular data: the rows-and-columns data you would normally keep in a spreadsheet. Instead of training a model from scratch on your dataset, it uses a single pretrained transformer that makes predictions in one forward pass. In practice you get the familiar scikit-learn workflow, `fit` then `predict`, with strong results in seconds and no tuning.

The quickest way to start is the **hosted API client** (`tabpfn-client`): the model runs on Prior Labs' servers, so there is nothing to download and no GPU to manage. We use the client throughout, then show at the end that the exact same code runs locally with the open-source `tabpfn` package.

We will look at the two most common prediction tasks:

- **Classification**: predicting a category, here whether a customer loan will default or not.
- **Regression**: predicting a number, here a measure of diabetes progression, together with the full range of likely values.

We start with setup and the one-time license step, then work through each task in turn.

tabpfn_architecture.svg

## Setup

*Installing the TabPFN client and scikit-learn.*

`tabpfn-client` is the hosted API client; `tabpfn` is the open-source package we use for the local example at the end. scikit-learn supplies the datasets and metrics used throughout.

In [ ]:
!pip install tabpfn tabpfn-client scikit-learn

## License and Authentication

*Accept the license once, then set a token so the client can reach the API.*

The first time you use TabPFN you accept a one-time license and get an API token:

1. Open [priorlabs.ai](https://ux.priorlabs.ai) and log in or register.
2. Accept the license on the Licenses tab.
3. Copy your API key and set it as the `TABPFN_TOKEN` environment variable.

The client reads `TABPFN_TOKEN` to authenticate, and the same token later lets the local `tabpfn` package download its weights. Here we read it from Colab secrets. Setting it before the first `fit` means everything below runs without interruption.

In [ ]:
import os
from google.colab import userdata
from tabpfn_client import set_access_token

os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")
set_access_token(userdata.get('TABPFN_TOKEN'))

## Binary Classification

*Fitting `TabPFNClassifier` on the German credit dataset.*

We fetch the German Credit dataset directly from OpenML and split it with stratification so the class balance is preserved in both halves. We then evaluate the held-out predictions with ROC AUC and accuracy.

**What `fit` does, and does not, do.** With most models, `fit` is where training happens: weights are updated by gradient descent over many passes through the data. TabPFN is different. Its transformer was pretrained once by Prior Labs and the weights stay frozen, so `fit` does not train or fine-tune anything. With the client, `fit` simply sends your training data to the hosted model, which keeps it as the context it reads at prediction time. The real work happens in a single forward pass during `predict`.

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from tabpfn_client import TabPFNClassifier

X, y = fetch_openml(data_id=46562, as_frame=True, return_X_y=True)

In [ ]:
print(X.shape, y.shape)

(1000, 33) (1000,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42, stratify=y
)

In [ ]:
clf = TabPFNClassifier()
clf.fit(X_train, y_train)

00:00 Fitting... -

00:00 Fitting... Done!


TabPFNClassifier(client_options=ClientOptions(timeout=900.0,
                                              headers={'sentry-trace': '008217a67ab64adc92286ca403317973'}))

In [ ]:
prediction_probabilities = clf.predict_proba(X_test)
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))

00:00 Predicting... -

00:01 Predicting... Done!
ROC AUC: 0.8050636232454415


In [ ]:
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))

00:00 Predicting... -

00:01 Predicting... Done!
Accuracy 0.7818181818181819


## Regression

*The same fit/predict workflow, now with `TabPFNRegressor`.*

Regression follows an identical pattern. We load the diabetes dataset and fit `TabPFNRegressor`. Beyond a single point estimate, TabPFN models the full predictive distribution, so we can also request specific quantiles or the distribution's mode.

### Load the data

*The diabetes regression dataset, split into train and test.*

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tabpfn_client import TabPFNRegressor

X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.33,
    random_state=42,
)

### Instantiate the regressor

*Fitting downloads the regressor weights on first use.*

In [ ]:
reg = TabPFNRegressor()
reg.fit(X_train, y_train)

00:00 Fitting... -

00:00 Fitting... Done!


TabPFNRegressor(client_options=ClientOptions(timeout=900.0,
                                             headers={'sentry-trace': '7c4a59b6e12149539eb96ae5f4cd8ff2'}))

### Point predictions

*The default output: the mean of the predictive distribution.*

We report MSE, MAE, and R-squared on the test set.

In [ ]:
predictions = reg.predict(X_test)
print("Mean Squared Error (MSE):", mean_squared_error(y_test, predictions))
print("Mean Absolute Error (MAE):", mean_absolute_error(y_test, predictions))
print("R-squared (R^2):", r2_score(y_test, predictions))

00:00 Predicting... -

00:00 Predicting... Done!
Mean Squared Error (MSE): 2698.666027142602
Mean Absolute Error (MAE): 40.93228039676196
R-squared (R^2): 0.5310957151096967


## Running Locally

*The same code on the open-source `tabpfn` package.*

Everything above ran on the hosted API. The open-source `tabpfn` package exposes the identical scikit-learn interface, so the only change is the import. The first `fit` downloads the pretrained weights (a few hundred MB) and every prediction then runs on your own CPU or GPU, with no data leaving your machine.

In [ ]:
from tabpfn import TabPFNClassifier

X, y = fetch_openml(data_id=46562, as_frame=True, return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42, stratify=y
)

clf = TabPFNClassifier()  # downloads weights on first use, then runs locally
clf.fit(X_train, y_train)
print("Local ROC AUC:", roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1]))

/usr/local/lib/python3.12/dist-packages/tabpfn/validation.py:142: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  _validate_num_samples_for_cpu(


Local ROC AUC: 0.8055446237264419
